# Machine Learning #10 — Object Detection & Localization in PyTorch 📦🔎

So far our image models answered one question: **what** is in the picture? Today we add a
second one: **where** is it — and what if there are **many** objects?

We build directly on three earlier labs:

- **#02** — classification *and* regression losses → a localizer needs **both at once**,
- **#03** — CNNs → the feature extractor,
- **#04** — transfer learning → we fine-tune a **pre-trained detector** by swapping its head.

### Our workflow today

> **boxes → IoU → NMS → AP → one-object localizer → pre-trained detector → fine-tuning on pedestrians**

The notebook has five parts:

1. ✋ **The three calculations of detection** — IoU, non-maximum suppression and average
   precision, by hand and in code.
2. 🎯 **Single-object localization** — one CNN predicts a class **and** a box.
3. 🏗️ **How real detectors work** — anchors, one-stage vs. two-stage, and transformers.
4. 🚀 **A pre-trained detector** — Faster R-CNN trained on COCO.
5. 🚶 **Fine-tuning** that detector on the Penn-Fudan pedestrian dataset.

##### ⚠️ Remember to set the **GPU** runtime in Colab! (`Runtime → Change runtime type → T4 GPU`)

# 📒 What is object detection?

| Task | Question | Output for one image |
|---|---|---|
| **Classification** | *What* is in the image? | one label |
| **Localization** | What is it, and *where* is it? | one label + **one box** |
| **Object detection** | Which objects are there, and where is *each* of them? | a **variable-length list** of (label, box, score) |
| **Instance segmentation** | …and exactly which pixels belong to each object? | a list of (label, mask, score) |

Detection is harder than classification for three reasons:

1. **The number of outputs is not fixed.** An image can contain no pedestrian or twenty.
2. **Objects appear at very different sizes** — a face filling the picture vs. a person far away.
3. **Two tasks at once:** recognising the class (*what*) and regressing a box (*where*).

Typical applications: pedestrians, vehicles and traffic signs in driver assistance, defects in
visual inspection, lesions or cells in medical images, products on shelves, animals in
camera-trap photos, and tables or figures in scanned documents.

## 📐 How do we describe a box?

A box is four numbers, but there are **several conventions** — and mixing them up is one of the
most common bugs in detection code. 🐛

| Format | The four numbers | Used by |
|---|---|---|
| `xyxy` | $(x_{min},\ y_{min},\ x_{max},\ y_{max})$ — two corners | Pascal VOC, **torchvision** |
| `xywh` | $(x_{min},\ y_{min},\ w,\ h)$ — corner + size | COCO annotation files |
| `cxcywh` | $(c_x,\ c_y,\ w,\ h)$ — centre + size | YOLO label files (normalised to $[0, 1]$), DETR |

Image coordinates start in the **top-left corner** and $y$ grows **downwards**.

$$\text{area} = (x_{max} - x_{min}) \cdot (y_{max} - y_{min})$$

## ⚙️ Imports and setup

Everything today comes from `torch` and `torchvision`; the sample photos come from
`scikit-image`. All three are preinstalled in Colab and listed in the repository's
`pyproject.toml`.

In [ ]:
import random
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from skimage import data as skdata

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
from torchvision import tv_tensors
from torchvision.io import read_image, ImageReadMode
from torchvision.ops import box_iou, box_convert, nms, batched_nms, masks_to_boxes
from torchvision.transforms import v2 as T
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_fpn,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch version      :", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("device             :", device)

### Two drawing helpers we will reuse everywhere 🔧

- `to_display(image)` — turns a PyTorch image `(3, H, W)` into the `(H, W, 3)` layout that
  `plt.imshow` expects.
- `draw_boxes(ax, boxes, ...)` — draws `xyxy` boxes (in pixels) with optional text labels.

In [ ]:
def to_display(image):
    # (3, H, W) tensor (float in [0, 1] or uint8)  ->  (H, W, 3) NumPy array for plt.imshow
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().as_subclass(torch.Tensor)
        if image.dtype == torch.uint8:
            image = image.float() / 255
        image = image.permute(1, 2, 0).numpy()
    return image


def draw_boxes(ax, boxes, color="red", labels=None, linestyle="-", linewidth=2):
    # boxes: (N, 4) in xyxy pixel coordinates
    boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
    for i, (x1, y1, x2, y2) in enumerate(boxes.tolist()):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=color, linewidth=linewidth, linestyle=linestyle))
        if labels is not None:
            ax.text(x1, y1, labels[i], color="white", fontsize=8, va="bottom",
                    bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))

### 🔁 Converting between formats

`torchvision.ops.box_convert` converts between the three formats. Let's convert one box and
check the numbers by hand: corner $(40, 30)$, width $80$, height $60$.

In [ ]:
box_xyxy = torch.tensor([[40.0, 30.0, 120.0, 90.0]])

print("xyxy  :", box_xyxy.tolist())
print("xywh  :", box_convert(box_xyxy, in_fmt="xyxy", out_fmt="xywh").tolist())
print("cxcywh:", box_convert(box_xyxy, in_fmt="xyxy", out_fmt="cxcywh").tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.8))
ax.imshow(np.ones((120, 160, 3)))
draw_boxes(ax, box_xyxy, color="tab:blue")

ax.plot([40, 120], [30, 90], "o", color="tab:blue")
ax.annotate("(x_min, y_min) = (40, 30)", (40, 30), xytext=(2, 14), fontsize=9)
ax.annotate("(x_max, y_max) = (120, 90)", (120, 90), xytext=(62, 108), fontsize=9)
ax.plot(80, 60, "x", color="tab:red", markersize=10)
ax.annotate("centre (80, 60)", (80, 60), xytext=(58, 52), fontsize=9, color="tab:red")

ax.set_title("x grows to the right, y grows DOWN")
plt.tight_layout()
plt.show()

---
# ✋ Part 1 — The three calculations of detection

Every detector — old or new — is trained and evaluated with three small calculations:
**IoU**, **non-maximum suppression** and **average precision**. You will meet them in every
detection paper (and in the exam ✍️).

## 📏 1. Intersection over Union (IoU)

How well does a predicted box match the true one? We measure the **overlap relative to the
total area that the two boxes cover**:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

- IoU $= 1$ → identical boxes, IoU $= 0$ → no overlap at all.
- The intersection of two axis-aligned boxes is again a box:

$$x_1 = \max(x^A_{min}, x^B_{min}),\quad y_1 = \max(y^A_{min}, y^B_{min}),\quad
x_2 = \min(x^A_{max}, x^B_{max}),\quad y_2 = \min(y^A_{max}, y^B_{max})$$

$$|A \cap B| = \max(0,\ x_2 - x_1) \cdot \max(0,\ y_2 - y_1)$$

The $\max(0, \cdot)$ handles boxes that do not overlap — their "intersection" would have a
negative width.

### ✍️ By hand

$A = (2, 2, 6, 6)$ and $B = (4, 4, 8, 8)$ in `xyxy`:

1. intersection box: $(\max(2,4),\ \max(2,4),\ \min(6,8),\ \min(6,8)) = (4, 4, 6, 6)$ → area $2 \cdot 2 = 4$
2. areas: $|A| = 4 \cdot 4 = 16$, $\ |B| = 16$
3. union: $16 + 16 - 4 = 28$
4. IoU $= 4 / 28 \approx \mathbf{0.143}$

Two boxes that look "fairly close" have an IoU of only 0.14. Pascal VOC counts a detection as
correct only from **IoU ≥ 0.5**; COCO additionally uses stricter thresholds up to 0.95.

### 💻 IoU in code — for many boxes at once

We compute the IoU between **every** box in `boxes_a` (N boxes) and **every** box in `boxes_b`
(M boxes) without a Python loop, using broadcasting: `(N, 1, 2)` against `(1, M, 2)` gives `(N, M, 2)`.

In [ ]:
def iou_pairwise(boxes_a, boxes_b):
    # boxes_a: (N, 4), boxes_b: (M, 4), both xyxy with positive area  ->  IoU matrix (N, M)
    boxes_a = torch.as_tensor(boxes_a, dtype=torch.float32).reshape(-1, 4)
    boxes_b = torch.as_tensor(boxes_b, dtype=torch.float32).reshape(-1, 4)

    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])     # (N,)
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])     # (M,)

    top_left     = torch.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])       # (N, M, 2)
    bottom_right = torch.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])       # (N, M, 2)
    wh = (bottom_right - top_left).clamp(min=0)          # negative width/height -> no overlap
    intersection = wh[..., 0] * wh[..., 1]               # (N, M)

    union = area_a[:, None] + area_b[None, :] - intersection
    return intersection / union

In [ ]:
A = torch.tensor([[2.0, 2.0, 6.0, 6.0]])
B = torch.tensor([[4.0, 4.0, 8.0, 8.0]])

print(f"our IoU         : {iou_pairwise(A, B).item():.4f}")
print(f"torchvision IoU : {box_iou(A, B).item():.4f}")
print(f"by hand         : {4 / 28:.4f}")

In [ ]:
# A stronger check: 50 x 40 random boxes against torchvision's implementation
def random_boxes(n, generator):
    xy = torch.rand(n, 2, generator=generator) * 100
    wh = torch.rand(n, 2, generator=generator) * 50 + 1        # +1: no zero-size boxes
    return torch.cat([xy, xy + wh], dim=1)

g = torch.Generator().manual_seed(SEED)
boxes_1, boxes_2 = random_boxes(50, g), random_boxes(40, g)

ours = iou_pairwise(boxes_1, boxes_2)
print("shape:", tuple(ours.shape))
print("same as torchvision:", torch.allclose(ours, box_iou(boxes_1, boxes_2), atol=1e-6))
print("pairs that do not overlap at all:", int((ours == 0).sum()), "of", ours.numel())

### 👀 What does an IoU of 0.5 look like?

Below we shift a $100 \times 100$ box diagonally by a few pixels. People tend to
**overestimate** overlap: a box that looks like a decent match may have an IoU close to 0.5.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
truth = torch.tensor([[50.0, 50.0, 150.0, 150.0]])

for ax, shift in zip(axes, [5, 15, 30]):
    shifted = truth + shift
    ax.imshow(np.ones((200, 200, 3)))
    draw_boxes(ax, truth, color="lime", linestyle="--")
    draw_boxes(ax, shifted, color="red")
    ax.set_title(f"shift {shift} px  ->  IoU = {iou_pairwise(truth, shifted).item():.2f}")
    ax.axis("off")

plt.suptitle("green = ground truth, red = prediction")
plt.tight_layout()
plt.show()

### 🔎 Small objects are much harder

Shift a square of side $s$ by $d$ pixels to the right. The intersection is $(s - d) \cdot s$
and the union is $(s + d) \cdot s$, so

$$\text{IoU} = \frac{s - d}{s + d}.$$

The **same pixel error** costs a small box much more IoU than a large box. This is one reason
why detectors are usually weakest on small objects, and why COCO reports AP separately for
small, medium and large objects.

In [ ]:
shifts = np.arange(0, 31)

plt.figure(figsize=(6.5, 4))
for side in [16, 64, 256]:
    plt.plot(shifts, (side - shifts) / (side + shifts), label=f"box side {side} px")
plt.axhline(0.5, color="gray", linestyle="--", label="IoU = 0.5")
plt.xlabel("horizontal shift of the prediction (pixels)")
plt.ylabel("IoU with the ground truth")
plt.ylim(0, 1.02)
plt.legend()
plt.tight_layout()
plt.show()

## 🧹 2. Non-maximum suppression (NMS)

A detector does not output one box per object. It scores **thousands of candidate boxes**, and
neighbouring candidates often fire on the **same object** → many near-duplicate boxes.

**Greedy NMS** keeps the most confident box and removes its duplicates:

1. Sort the boxes by score, highest first.
2. **Keep** the highest-scoring remaining box.
3. **Remove** every remaining box whose IoU with it is **above** a threshold (e.g. 0.5).
4. Repeat from step 2 until no boxes are left.

NMS is a hand-written rule applied **after** the network — it has no trainable parameters.

Our example: two objects and six candidate boxes.

| box | `xyxy` | score |
|---|---|---|
| A | (10, 10, 60, 70) | 0.92 |
| B | (14, 6, 64, 66) | 0.81 |
| C | (6, 16, 56, 74) | 0.60 |
| D | (80, 20, 130, 80) | 0.88 |
| E | (76, 24, 124, 84) | 0.70 |
| F | (45, 30, 95, 75) | 0.55 |

In [ ]:
nms_names  = ["A", "B", "C", "D", "E", "F"]
nms_boxes  = torch.tensor([[10, 10, 60, 70],
                           [14,  6, 64, 66],
                           [ 6, 16, 56, 74],
                           [80, 20, 130, 80],
                           [76, 24, 124, 84],
                           [45, 30, 95, 75]], dtype=torch.float32)
nms_scores = torch.tensor([0.92, 0.81, 0.60, 0.88, 0.70, 0.55])

# The full IoU table - useful for checking a hand calculation
pd.DataFrame(iou_pairwise(nms_boxes, nms_boxes).numpy().round(3), index=nms_names, columns=nms_names)

In [ ]:
def nms_greedy(boxes, scores, iou_threshold=0.5, names=None):
    # Returns the indices of the kept boxes, highest score first.
    # If `names` is given, every step is printed.
    boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
    scores = torch.as_tensor(scores, dtype=torch.float32).reshape(-1)

    order = torch.argsort(scores, descending=True)     # step 1
    keep = []
    while order.numel() > 0:
        best = order[0].item()                         # step 2: the most confident remaining box
        keep.append(best)
        rest = order[1:]
        ious = iou_pairwise(boxes[best], boxes[rest])[0]
        if names is not None:
            removed = [f"{names[i]} (IoU {v:.2f})"
                       for i, v in zip(rest.tolist(), ious.tolist()) if v > iou_threshold]
            print(f"keep {names[best]} (score {scores[best]:.2f})  ->  remove: {', '.join(removed) or '-'}")
        order = rest[ious <= iou_threshold]            # step 3: drop the duplicates of `best`
    return torch.tensor(keep, dtype=torch.long)

In [ ]:
kept = nms_greedy(nms_boxes, nms_scores, iou_threshold=0.5, names=nms_names)

print("\nours       :", [nms_names[i] for i in kept.tolist()])
print("torchvision:", [nms_names[i] for i in nms(nms_boxes, nms_scores, iou_threshold=0.5).tolist()])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
labels_all = [f"{n} {s:.2f}" for n, s in zip(nms_names, nms_scores.tolist())]

for ax, idx, title in [(axes[0], torch.arange(len(nms_boxes)), "before NMS: 6 boxes"),
                       (axes[1], kept, f"after NMS (IoU > 0.5 removed): {len(kept)} boxes")]:
    ax.imshow(np.ones((95, 140, 3)))
    draw_boxes(ax, nms_boxes[idx], color="red", labels=[labels_all[i] for i in idx.tolist()])
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

> 🔎 **Box F survives.** It overlaps both objects, but not enough with either to be called a
> duplicate. NMS removes **duplicates**, not **false positives** — removing F is the job of the
> score threshold (or of a better model).

## 🏷️ Class-aware NMS

NMS should only remove duplicates of the **same object**. Picture a child sitting on a chair:
the `person` box and the `chair` box overlap heavily, yet both are correct.

| box | `xyxy` | class | score |
|---|---|---|---|
| child | (60, 40, 140, 170) | person | 0.90 |
| chair | (50, 60, 150, 180) | chair | 0.85 |

Their IoU is about 0.65, so **class-agnostic** NMS with threshold 0.5 deletes the chair.
**Class-aware** NMS runs NMS separately for each class.

> 💡 `torchvision.ops.batched_nms` gets the same result with one trick: it shifts the boxes of
> each class by a large class-dependent offset, so boxes of different classes can never overlap,
> and then runs NMS once.

In [ ]:
def nms_per_class(boxes, scores, labels, iou_threshold=0.5):
    # Class-aware NMS: run NMS for each class separately, then merge (highest score first)
    keep = []
    for c in torch.unique(labels):
        idx = torch.where(labels == c)[0]
        keep.append(idx[nms_greedy(boxes[idx], scores[idx], iou_threshold)])
    keep = torch.cat(keep)
    return keep[torch.argsort(scores[keep], descending=True)]


scene_boxes  = torch.tensor([[60.0, 40.0, 140.0, 170.0],     # child (person)
                             [50.0, 60.0, 150.0, 180.0]])    # chair
scene_scores = torch.tensor([0.90, 0.85])
scene_labels = torch.tensor([1, 2])                          # 1 = person, 2 = chair

print(f"IoU(child, chair)    = {iou_pairwise(scene_boxes[0], scene_boxes[1]).item():.3f}")
print("class-agnostic NMS   :", nms_greedy(scene_boxes, scene_scores, 0.5).tolist())
print("class-aware NMS      :", nms_per_class(scene_boxes, scene_scores, scene_labels, 0.5).tolist())
print("torchvision batched  :", batched_nms(scene_boxes, scene_scores, scene_labels, 0.5).tolist())

In [ ]:
# Random test: both NMS versions agree with torchvision on 200 random boxes
g = torch.Generator().manual_seed(SEED)
rand_boxes = random_boxes(200, g)
rand_scores = torch.rand(200, generator=g)
rand_labels = torch.randint(0, 3, (200,), generator=g)

for threshold in [0.3, 0.5, 0.7]:
    same_nms = torch.equal(nms_greedy(rand_boxes, rand_scores, threshold),
                           nms(rand_boxes, rand_scores, threshold))
    same_batched = torch.equal(nms_per_class(rand_boxes, rand_scores, rand_labels, threshold),
                               batched_nms(rand_boxes, rand_scores, rand_labels, threshold))
    print(f"threshold {threshold}: nms equal = {same_nms}, class-aware equal = {same_batched}")

## ✏️ Activity 1 — IoU and NMS with pen and paper ✍️

Do this **on paper first**, then check yourself with the cell below.

| box | `xyxy` | score |
|---|---|---|
| P | (0, 0, 10, 10) | 0.9 |
| Q | (1, 1, 11, 11) | 0.8 |
| R | (5, 0, 15, 10) | 0.7 |
| S | (20, 20, 30, 30) | 0.6 |

1. Compute IoU(P, Q), IoU(P, R), IoU(Q, R) and IoU(P, S).
2. Run greedy NMS with threshold **0.5**. Which boxes are kept?
3. Run it again with threshold **0.3**. What changes?
4. Suppose R is actually a **second person** standing right next to P. Which threshold is
   safer, and what is the price of choosing it?

In [ ]:
act_names  = ["P", "Q", "R", "S"]
act_boxes  = torch.tensor([[0, 0, 10, 10], [1, 1, 11, 11], [5, 0, 15, 10], [20, 20, 30, 30]],
                          dtype=torch.float32)
act_scores = torch.tensor([0.9, 0.8, 0.7, 0.6])

# Check your hand calculation here, e.g. with iou_pairwise(...) and nms_greedy(..., names=act_names)

## 📊 3. Precision, recall and average precision (AP)

How do we score a detector on a **whole dataset**? Three steps.

**Step 1 — matching.** Go through all predictions from the highest to the lowest score.
A prediction is a **true positive (TP)** if its IoU with a ground-truth box of the same class
is ≥ the threshold and that ground-truth box **has not been matched yet**. Otherwise it is a
**false positive (FP)** — a wrong location, or a duplicate of an object that was already found.
Ground-truth boxes that never get matched are **false negatives (FN)**.

**Step 2 — walk down the ranked list** and after each prediction compute

$$\text{precision} = \frac{TP}{TP + FP} \qquad\qquad \text{recall} = \frac{TP}{\text{number of ground-truth boxes}}$$

**Step 3 — AP** is the area under the precision–recall curve. We first replace every precision
by the **best precision reached at this recall or any higher recall** (the *envelope*), so the
curve never goes up again, and then add up rectangles.

### ✍️ By hand: 4 ground-truth objects, 6 predictions (already matched)

| rank | score | TP/FP | cum. TP | cum. FP | precision | recall | envelope |
|---|---|---|---|---|---|---|---|
| 1 | 0.95 | TP | 1 | 0 | 1/1 = 1.000 | 1/4 = 0.25 | 1.000 |
| 2 | 0.90 | FP | 1 | 1 | 1/2 = 0.500 | 0.25 | 0.667 |
| 3 | 0.80 | TP | 2 | 1 | 2/3 = 0.667 | 0.50 | 0.667 |
| 4 | 0.70 | FP | 2 | 2 | 2/4 = 0.500 | 0.50 | 0.600 |
| 5 | 0.60 | TP | 3 | 2 | 3/5 = 0.600 | 0.75 | 0.600 |
| 6 | 0.40 | FP | 3 | 3 | 3/6 = 0.500 | 0.75 | 0.500 |

Recall rises only at the TPs (ranks 1, 3, 5), each time by $0.25$:

$$\text{AP} = 0.25 \cdot 1.000 + 0.25 \cdot 0.667 + 0.25 \cdot 0.600 = \mathbf{0.567}$$

The fourth object was never detected, so recall stops at 0.75 and the last quarter of the
recall axis contributes **nothing**. Missed objects directly cap the AP.

| Name | Meaning |
|---|---|
| **AP50** | AP with IoU threshold 0.5 (the Pascal VOC metric) |
| **AP75** | AP with IoU threshold 0.75 — rewards precise boxes |
| **AP@[.50:.95]** | mean AP over the IoU thresholds 0.50, 0.55, …, 0.95 — the main **COCO** metric, often written just "AP" |
| **mAP** | AP averaged over **classes** (and, for COCO, over IoU thresholds) |

> ⚠️ Benchmarks differ in details — Pascal VOC 2007 used 11 recall points, later VOC uses all
> points (as we do), COCO uses 101 points and also limits detections per image. Numbers from
> different evaluation tools are therefore not always comparable to the last decimal.

In [ ]:
def precision_recall_ap(is_tp, scores, n_ground_truth):
    # is_tp: 1 for TP, 0 for FP (one entry per prediction); returns precision, recall, envelope, AP
    if n_ground_truth <= 0:
        raise ValueError("AP is undefined without ground-truth boxes")
    is_tp = np.asarray(is_tp, dtype=float)
    scores = np.asarray(scores, dtype=float)

    order = np.argsort(-scores, kind="stable")             # highest score first
    cum_tp = np.cumsum(is_tp[order])
    cum_fp = np.cumsum(1 - is_tp[order])

    precision = cum_tp / (cum_tp + cum_fp)
    recall = cum_tp / n_ground_truth

    # envelope: the best precision at this recall or at any higher recall (read from the right)
    envelope = np.maximum.accumulate(precision[::-1])[::-1]

    # area under the step curve: (increase of recall) x (envelope precision)
    recall_increase = np.diff(np.concatenate([[0.0], recall]))
    ap = float(np.sum(recall_increase * envelope))
    return precision, recall, envelope, ap

In [ ]:
example_tp     = [1, 0, 1, 0, 1, 0]
example_scores = [0.95, 0.90, 0.80, 0.70, 0.60, 0.40]

precision, recall, envelope, ap = precision_recall_ap(example_tp, example_scores, n_ground_truth=4)

print(pd.DataFrame({"score": example_scores, "TP": example_tp, "precision": precision.round(3),
                    "recall": recall.round(3), "envelope": envelope.round(3)}).to_string(index=False))
print(f"\nAP = {ap:.3f}")

In [ ]:
plt.figure(figsize=(5.5, 4))
plt.plot(recall, precision, "o", label="precision after each prediction")
plt.step(np.concatenate([[0], recall]), np.concatenate([[envelope[0]], envelope]),
         where="pre", label="envelope")
plt.fill_between(np.concatenate([[0], recall]), np.concatenate([[envelope[0]], envelope]),
                 step="pre", alpha=0.2, label=f"area = AP = {ap:.3f}")
plt.xlim(0, 1.02); plt.ylim(0, 1.05)
plt.xlabel("recall"); plt.ylabel("precision")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 💻 From boxes to AP

`match_predictions` performs **step 1** for one image: the most confident predictions claim
ground-truth boxes first, and every ground-truth box can be claimed only once.
`average_precision` collects the TP/FP flags of all images and calls `precision_recall_ap`.
`detection_report` evaluates AP50, AP75 and AP@[.50:.95].

We evaluate **one class** at a time; mAP would average these numbers over classes.

In [ ]:
def match_predictions(pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5):
    # One image, one class. Returns a 0/1 tensor: is each prediction a true positive?
    is_tp = torch.zeros(len(pred_boxes))
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return is_tp
    ious = iou_pairwise(pred_boxes, gt_boxes)                      # (predictions, ground truth)
    matched = torch.zeros(len(gt_boxes), dtype=torch.bool)
    for p in torch.argsort(pred_scores, descending=True).tolist():
        candidates = ious[p].clone()
        candidates[matched] = -1                                   # already claimed -> unavailable
        best = int(candidates.argmax())
        if candidates[best] >= iou_threshold:
            is_tp[p] = 1
            matched[best] = True
    return is_tp


def average_precision(predictions, targets, iou_threshold=0.5):
    # predictions: list of dicts with "boxes" and "scores"; targets: list of dicts with "boxes"
    flags, scores = [], []
    for pred, target in zip(predictions, targets, strict=True):
        flags.append(match_predictions(pred["boxes"], pred["scores"], target["boxes"], iou_threshold))
        scores.append(pred["scores"])
    n_ground_truth = sum(len(t["boxes"]) for t in targets)
    return precision_recall_ap(torch.cat(flags).numpy(), torch.cat(scores).numpy(), n_ground_truth)[3]


IOU_THRESHOLDS = np.round(np.linspace(0.50, 0.95, 10), 2)          # 0.50, 0.55, ..., 0.95

def detection_report(predictions, targets):
    aps = {t: average_precision(predictions, targets, t) for t in IOU_THRESHOLDS}
    return {"AP50": aps[0.50], "AP75": aps[0.75], "AP@[.50:.95]": float(np.mean(list(aps.values())))}

In [ ]:
# Sanity check on a tiny "dataset" of two images
toy_targets = [{"boxes": torch.tensor([[0., 0., 10., 10.], [20., 20., 30., 30.]])},
               {"boxes": torch.tensor([[5., 5., 15., 15.]])}]
toy_predictions = [
    {"boxes": torch.tensor([[0., 0., 10., 10.], [1., 1., 10., 10.], [21., 21., 30., 31.]]),
     "scores": torch.tensor([0.9, 0.8, 0.7])},    # perfect, duplicate (FP), good
    {"boxes": torch.tensor([[40., 40., 50., 50.]]),
     "scores": torch.tensor([0.95])},             # wrong place (FP); the object is missed (FN)
]
print("TP flags, image 1:", match_predictions(toy_predictions[0]["boxes"], toy_predictions[0]["scores"],
                                               toy_targets[0]["boxes"]).tolist())
detection_report(toy_predictions, toy_targets)

> 🤔 **Think about it:** in the toy example, the duplicate box has IoU 0.81 with the object,
> yet it counts as a **false positive**. Why is that the right decision — and which step of a
> detector is supposed to remove such duplicates before evaluation?

---
# 🎯 Part 2 — Single-object localization: classification + regression in one network

In ML #02 we built a classifier (cross-entropy) and a regressor (MSE) as **two separate
networks**. A **localizer** is both at once — one shared CNN with **two heads**:

- a **classification head** → class logits → cross-entropy,
- a **box head** → 4 numbers → a regression loss.

$$\mathcal{L} = \mathcal{L}_{\text{cls}} + \lambda \, \mathcal{L}_{\text{box}}$$

The weight $\lambda$ balances the two tasks (you will tune it in Activity 2).

We use a **synthetic dataset**: one random shape (square, circle or triangle) with a random
size, position and colour on a noisy background. Because we draw the shapes ourselves we know
the exact boxes, and the model trains in seconds even on a CPU — just like the Find-Max task in ML #08.

In [ ]:
SHAPES = ["square", "circle", "triangle"]
IMG_SIZE = 64


def shape_mask(shape, box, size=IMG_SIZE):
    # Boolean (size, size) mask of a shape that exactly fills box = (x1, y1, x2, y2)
    x1, y1, x2, y2 = box
    ys, xs = np.mgrid[0:size, 0:size] + 0.5                     # pixel centres
    cx, cy, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
    if shape == "square":
        return (xs >= x1) & (xs <= x2) & (ys >= y1) & (ys <= y2)
    if shape == "circle":
        return ((xs - cx) / (w / 2)) ** 2 + ((ys - cy) / (h / 2)) ** 2 <= 1
    if shape == "triangle":                                     # apex on top, base at the bottom
        half_width = (ys - y1) / h * (w / 2)
        return (ys >= y1) & (ys <= y2) & (np.abs(xs - cx) <= half_width)
    raise ValueError(f"unknown shape {shape!r}")


def make_shapes(n_images, n_objects=1, seed=SEED, size=IMG_SIZE):
    # Returns images (N, 3, size, size), boxes (N, n_objects, 4) in xyxy pixels, labels (N, n_objects)
    rng = np.random.default_rng(seed)
    images = np.empty((n_images, 3, size, size), dtype=np.float32)
    boxes = np.empty((n_images, n_objects, 4), dtype=np.float32)
    labels = np.empty((n_images, n_objects), dtype=np.int64)

    for i in range(n_images):
        image = np.repeat(rng.uniform(0.0, 0.35, size=(1, size, size)), 3, axis=0)   # grey noise
        for k in range(n_objects):
            while True:                                         # objects must not overlap
                side = rng.uniform(12, 36)
                x1, y1 = rng.uniform(0, size - side, size=2)
                box = np.array([x1, y1, x1 + side, y1 + side])
                if all(box[0] >= b[2] or b[0] >= box[2] or box[1] >= b[3] or b[1] >= box[3]
                       for b in boxes[i, :k]):
                    break
            shape = rng.integers(len(SHAPES))
            colour = rng.uniform(0.5, 1.0, size=3)              # colour says nothing about the class
            image[:, shape_mask(SHAPES[shape], box, size)] = colour[:, None]
            boxes[i, k], labels[i, k] = box, shape
        images[i] = image

    return torch.from_numpy(images), torch.from_numpy(boxes), torch.from_numpy(labels)

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(13, 4.8))
ex_images, ex_boxes, ex_labels = make_shapes(12, seed=0)

for ax, img, box, lab in zip(axes.flat, ex_images, ex_boxes, ex_labels):
    ax.imshow(to_display(img))
    draw_boxes(ax, box, color="lime", linewidth=1.5)
    ax.set_title(SHAPES[lab[0]], fontsize=10)
    ax.axis("off")
plt.suptitle("Synthetic localization data: one shape per image + its box")
plt.tight_layout()
plt.show()

### Train / validation / test 🔀

Each split is generated with its **own seed**, so no image appears in two splits.

The box target is stored as **normalised `cxcywh`** (divided by the image size, so all four
numbers lie in $[0, 1]$). The box head ends with a **sigmoid**, so its outputs lie in the same
range, and predicting a width and height (instead of two corners) guarantees a valid box.

In [ ]:
def to_localization_dataset(images, boxes, labels):
    box_targets = box_convert(boxes[:, 0], in_fmt="xyxy", out_fmt="cxcywh") / IMG_SIZE
    return TensorDataset(images, labels[:, 0], box_targets)


loc_train = to_localization_dataset(*make_shapes(3000, seed=SEED))
loc_val   = to_localization_dataset(*make_shapes(600,  seed=SEED + 1))
loc_test  = to_localization_dataset(*make_shapes(600,  seed=SEED + 2))

image, label, box = loc_train[0]
print("image:", tuple(image.shape), "| label:", SHAPES[label], "| box (cx, cy, w, h):", box.numpy().round(3))
print("train / val / test:", len(loc_train), len(loc_val), len(loc_test))

## 🧠 The localizer

The feature extractor is a small CNN like the one from ML #03. On top of it sit **two heads**
that see the same features.

> 💡 **Why `Flatten` and not global average pooling?** Global average pooling (Activity 6 in
> ML #04) averages each feature map over **all positions** — great for "what", but it throws
> away most of the information about "where". Flattening keeps the spatial layout, and the box
> head needs exactly that.

In [ ]:
class Localizer(nn.Module):
    def __init__(self, num_classes=len(SHAPES)):
        super().__init__()

        def block(c_in, c_out):
            return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out),
                                 nn.ReLU(), nn.MaxPool2d(2))

        self.features = nn.Sequential(
            block(3, 16), block(16, 32), block(32, 64), block(64, 64),   # 64 -> 32 -> 16 -> 8 -> 4
            nn.Flatten(),                                                 # 64 * 4 * 4 = 1024
        )
        self.cls_head = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Linear(128, num_classes))
        self.box_head = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Linear(128, 4), nn.Sigmoid())

    def forward(self, x):
        features = self.features(x)
        return self.cls_head(features), self.box_head(features)   # (logits, normalised cxcywh)


logits, boxes = Localizer()(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE))
print("class logits:", tuple(logits.shape), "| box:", tuple(boxes.shape))

## ⭐ Loss and training

- **Classification:** cross-entropy on the logits, exactly as in ML #02.
- **Box:** **L1 loss** (mean absolute error) between the predicted and true normalised box.
  Detectors traditionally use L1 or its smooth variant (*smooth L1* = L1 with a rounded,
  quadratic tip near zero) because it is less dominated by a few large errors than MSE.

The evaluation reports four numbers:

| metric | meaning |
|---|---|
| accuracy | correct class |
| mean IoU | average overlap between predicted and true box |
| IoU ≥ 0.5 | the box would count as a hit |
| class ✓ and IoU ≥ 0.5 | **both** right — the usual definition of a correct localization |

In [ ]:
@torch.no_grad()
def evaluate_localizer(model, dataset):
    model.eval()
    images, labels, box_targets = dataset.tensors
    logits, box_pred = model(images.to(device))
    pred_xyxy = box_convert(box_pred.cpu() * IMG_SIZE, in_fmt="cxcywh", out_fmt="xyxy")
    true_xyxy = box_convert(box_targets * IMG_SIZE, in_fmt="cxcywh", out_fmt="xyxy")

    ious = iou_pairwise(pred_xyxy, true_xyxy).diagonal()     # IoU of each image with ITS OWN box
    correct = logits.argmax(dim=1).cpu() == labels
    return {"accuracy": correct.float().mean().item(),
            "mean IoU": ious.mean().item(),
            "IoU >= 0.5": (ious >= 0.5).float().mean().item(),
            "class and IoU >= 0.5": (correct & (ious >= 0.5)).float().mean().item()}


def train_localizer(box_weight=5.0, epochs=15, lr=1e-3, batch_size=64, seed=SEED):
    torch.manual_seed(seed)
    model = Localizer().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(loc_train, batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    history = []

    for epoch in range(epochs):
        model.train()
        sums = {"cls": 0.0, "box": 0.0}
        for x, y, box in loader:
            x, y, box = x.to(device), y.to(device), box.to(device)
            logits, box_pred = model(x)
            loss_cls = F.cross_entropy(logits, y)
            loss_box = F.l1_loss(box_pred, box)
            loss = loss_cls + box_weight * loss_box          # one number -> one backward pass

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            sums["cls"] += loss_cls.item() * len(x)
            sums["box"] += loss_box.item() * len(x)

        val = evaluate_localizer(model, loc_val)
        history.append({"epoch": epoch + 1,
                        "train CE": sums["cls"] / len(loc_train),
                        "train L1": sums["box"] / len(loc_train), **val})
    return model, pd.DataFrame(history)

In [ ]:
%%time
localizer, loc_history = train_localizer(box_weight=5.0)
loc_history.round(3).tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(loc_history["epoch"], loc_history["train CE"], label="classification (CE)")
ax[0].plot(loc_history["epoch"], loc_history["train L1"], label="box (L1)")
ax[0].set_xlabel("epoch"); ax[0].set_title("training losses"); ax[0].legend()

for col in ["accuracy", "mean IoU", "class and IoU >= 0.5"]:
    ax[1].plot(loc_history["epoch"], loc_history[col], label=col)
ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1.02); ax[1].set_title("validation"); ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def predict_boxes(model, images):
    model.eval()
    logits, box_pred = model(images.to(device))
    return logits.argmax(dim=1).cpu(), box_convert(box_pred.cpu() * IMG_SIZE, "cxcywh", "xyxy")


val_images, val_labels, val_targets = loc_val[:12]
pred_labels, pred_xyxy = predict_boxes(localizer, val_images)
true_xyxy = box_convert(val_targets * IMG_SIZE, "cxcywh", "xyxy")

fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(to_display(val_images[i]))
    draw_boxes(ax, true_xyxy[i], color="lime", linestyle="--", linewidth=1.5)
    draw_boxes(ax, pred_xyxy[i], color="red", linewidth=1.5)
    iou = iou_pairwise(true_xyxy[i], pred_xyxy[i]).item()
    mark = "✓" if pred_labels[i] == val_labels[i] else "✗"
    ax.set_title(f"{SHAPES[pred_labels[i]]} {mark}  IoU {iou:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("validation: green dashed = truth, red = prediction")
plt.tight_layout()
plt.show()

## 🤔 What about two objects?

Our localizer has exactly **one** box output. What does it do with an image that contains
**two** shapes? (It never saw such images during training, so this is also a distribution shift.)

In [ ]:
two_images, two_boxes, two_labels = make_shapes(12, n_objects=2, seed=7)
pred_labels, pred_xyxy = predict_boxes(localizer, two_images)

fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(to_display(two_images[i]))
    draw_boxes(ax, two_boxes[i], color="lime", linestyle="--", linewidth=1.5)
    draw_boxes(ax, pred_xyxy[i], color="red", linewidth=1.5)
    best_iou = iou_pairwise(pred_xyxy[i], two_boxes[i]).max().item()
    ax.set_title(f"{SHAPES[pred_labels[i]]}, best IoU {best_iou:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("two objects, one output: green dashed = the two true boxes, red = the only prediction")
plt.tight_layout()
plt.show()

### 💡 The core problem of detection

- One output can report **at most one** object — the other is always missed. Sometimes the box
  lands **between** the two shapes and matches neither.
- We could give the network *K* box outputs — but how large should *K* be, and **which output
  is responsible for which object**? Assigning ground-truth objects to outputs is the central
  design question of every detector.

Real detectors answer it by predicting a **large, fixed set of candidate boxes**, each with a
score saying *"is there an object here?"*, and then cleaning up the duplicates. That is Part 3.

---
# 🏗️ Part 3 — How real detectors work

## ⚓ Dense candidates: anchors

Place many reference boxes — **anchors** — of several sizes and aspect ratios at every position
of a CNN feature map. For **every anchor** the network predicts:

1. a **score** — is there an object here (and of which class)?
2. a **box correction** — how to shift and scale the anchor so that it fits the object.

Training assigns targets with **IoU** again: an anchor that overlaps a ground-truth box enough
(e.g. IoU ≥ 0.7 in the Faster R-CNN proposal network) is a **positive** and learns that box;
anchors with little overlap (IoU < 0.3) are **negatives** (background); anchors in between are ignored.

The network predicts **offsets relative to the anchor**, not raw pixel coordinates — small,
well-scaled numbers that are much easier to regress than "pixel 437":

$$t_x = \frac{x - x_a}{w_a},\qquad t_y = \frac{y - y_a}{h_a},\qquad t_w = \ln\frac{w}{w_a},\qquad t_h = \ln\frac{h}{h_a}$$

where $(x, y, w, h)$ is the centre and size of the ground-truth box and $(x_a, y_a, w_a, h_a)$
that of the anchor. Thanks to the logarithm, *twice too large* and *twice too small* give
errors of the same size ($\ln 2$ vs. $-\ln 2$).

### ✍️ By hand

Anchor: centre $(100, 100)$, size $64 \times 64$. Ground truth: centre $(110, 96)$, size $80 \times 48$.

$$t_x = \tfrac{10}{64} = 0.156,\quad t_y = \tfrac{-4}{64} = -0.063,\quad
t_w = \ln\tfrac{80}{64} = 0.223,\quad t_h = \ln\tfrac{48}{64} = -0.288$$

In [ ]:
def encode(box, anchor):
    # box and anchor in (cx, cy, w, h)  ->  regression targets (tx, ty, tw, th)
    x, y, w, h = box
    xa, ya, wa, ha = anchor
    return np.array([(x - xa) / wa, (y - ya) / ha, np.log(w / wa), np.log(h / ha)])


def decode(t, anchor):
    # the inverse: offsets predicted by the network -> a box in (cx, cy, w, h)
    tx, ty, tw, th = t
    xa, ya, wa, ha = anchor
    return np.array([xa + tx * wa, ya + ty * ha, wa * np.exp(tw), ha * np.exp(th)])


anchor = np.array([100.0, 100.0, 64.0, 64.0])
gt_box = np.array([110.0, 96.0, 80.0, 48.0])

t = encode(gt_box, anchor)
print("targets (tx, ty, tw, th):", t.round(3))
print("decoded back            :", decode(t, anchor))

> 💡 torchvision additionally multiplies these offsets by constant weights (e.g. 10, 10, 5, 5
> in the second stage of Faster R-CNN) to balance the loss terms; the idea is the same.

## 🧭 Three families of detectors

| | **Two-stage** — e.g. Faster R-CNN | **One-stage** — e.g. SSD, RetinaNet, YOLO | **Set prediction** — DETR family |
|---|---|---|---|
| How | ① a *Region Proposal Network* scores anchors and proposes object-like regions; ② a second head classifies and refines each proposal | classifies and regresses every anchor or grid position in a single pass | a transformer decoder turns a fixed set of learned **object queries** into (class, box) pairs |
| Duplicates removed by | NMS | NMS (some recent versions are trained to avoid it) | **one-to-one matching** during training (Hungarian algorithm) → no NMS |
| Typically | accurate and flexible | fast, easy to deploy | no anchors or NMS to hand-tune |

Many modern one-stage detectors are **anchor-free**: they predict boxes relative to grid
points instead of anchors. The two ingredients stay the same: **dense candidates + a way to
handle duplicates**.

DETR closes the loop with ML #08 and #09: its object queries use **cross-attention** to look
at the image features. In Activity 5 you will try a DETR model through the 🤗 `pipeline` API.

## 🔬 Anatomy of Faster R-CNN (the model we use next)

```
image ─► backbone (MobileNetV3) ─► FPN: feature maps at several resolutions
      ─► RPN: objectness + box offsets for every anchor ─► NMS ─► top proposals
      ─► RoI head: crop features of each proposal (RoIAlign) ─► class scores + class-specific box refinement
      ─► post-processing: score threshold ─► class-aware NMS ─► at most 100 detections
```

The **FPN** (Feature Pyramid Network) provides features at several resolutions, so small objects
can be detected on fine feature maps and large objects on coarse ones.

---
# 🚀 Part 4 — A pre-trained detector: Faster R-CNN (COCO)

`torchvision` ships several detectors pre-trained on **COCO** (80 object categories, about
118 000 training images). We use **`fasterrcnn_mobilenet_v3_large_fpn`** — Faster R-CNN with a
light MobileNetV3 backbone and an FPN. Larger backbones (e.g. ResNet-50) are more accurate, but
slower; this one stays usable even without a GPU.

> 💡 The first run downloads about 75 MB of weights; later runs use the local cache.

In [ ]:
detector_weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
coco_detector = fasterrcnn_mobilenet_v3_large_fpn(weights=detector_weights).to(device)
coco_detector.eval()

COCO_CATEGORIES = detector_weights.meta["categories"]
print("weights:", detector_weights)
print("box AP on COCO val2017 reported by torchvision:",
      detector_weights.meta["_metrics"]["COCO-val2017"]["box_map"])
print("number of category entries:", len(COCO_CATEGORIES))
print(COCO_CATEGORIES[:16])

### Reading the category list

- index `0` is `__background__` — the "no object" class that every proposal can also be,
- entries named `N/A` fill gaps: the original COCO category ids are not contiguous
  (91 ids for 80 used categories), and torchvision keeps the original numbering.

### What does the model expect?

| | Classification models (ML #04) | torchvision detection models |
|---|---|---|
| input | one batch tensor `(N, 3, 224, 224)`, normalised by us | a **list** of `(3, H, W)` float tensors in $[0, 1]$, **any size** |
| preprocessing | `weights.transforms()` resizes + normalises | the model resizes and normalises **internally** |
| output | logits `(N, classes)` | a **list of dicts** with `boxes` (`xyxy`, original pixels), `labels`, `scores` |

The internal preprocessing is visible in `model.transform`:

In [ ]:
coco_detector.transform

### 📦 Today's fine-tuning dataset — downloaded now

We download the **Penn-Fudan pedestrian dataset** (about 54 MB) already here, because its street
scenes are perfect test images for the detector. The download happens only once.

In [ ]:
DATA_DIR = Path("data")
PENNFUDAN_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
PENNFUDAN_DIR = DATA_DIR / "PennFudanPed"

if len(list((PENNFUDAN_DIR / "PNGImages").glob("*.png"))) != 170:
    DATA_DIR.mkdir(exist_ok=True)
    zip_path = DATA_DIR / "PennFudanPed.zip"
    print("downloading Penn-Fudan (about 54 MB) ...")
    urllib.request.urlretrieve(PENNFUDAN_URL, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATA_DIR)
    zip_path.unlink()

print(sorted(p.name for p in PENNFUDAN_DIR.iterdir()))
print("images:", len(list((PENNFUDAN_DIR / "PNGImages").glob("*.png"))))

## 🔮 Run the detector

`detect` wraps the three steps: move the images to the device, run the model in evaluation
mode without gradients, and move the results back to the CPU.

In [ ]:
@torch.no_grad()
def detect(model, images):
    # images: list of (3, H, W) float tensors in [0, 1]  ->  list of dicts on the CPU
    model.eval()
    outputs = model([img.to(device) for img in images])
    return [{k: v.cpu() for k, v in out.items()} for out in outputs]


def numpy_to_tensor(array):
    # (H, W, 3) uint8 NumPy image -> (3, H, W) float tensor in [0, 1]
    return torch.from_numpy(array).permute(2, 0, 1).float() / 255


demo_images = {
    "astronaut": numpy_to_tensor(skdata.astronaut()),
    "cat":       numpy_to_tensor(skdata.chelsea()),
    "coffee":    numpy_to_tensor(skdata.coffee()),
    "street":    read_image(str(PENNFUDAN_DIR / "PNGImages" / "FudanPed00025.png"),
                            mode=ImageReadMode.RGB).float() / 255,
}
demo_outputs = dict(zip(demo_images, detect(coco_detector, list(demo_images.values()))))

for name, image in demo_images.items():
    print(f"{name:10s} image {tuple(image.shape)} -> {len(demo_outputs[name]['boxes'])} detections")

In [ ]:
# The raw output for the street image: boxes, labels and scores (sorted by score)
street = demo_outputs["street"]
print({k: tuple(v.shape) for k, v in street.items()})

pd.DataFrame({
    "category": [COCO_CATEGORIES[i] for i in street["labels"].tolist()],
    "score": street["scores"].numpy().round(3),
    "x_min": street["boxes"][:, 0].numpy().round(1), "y_min": street["boxes"][:, 1].numpy().round(1),
    "x_max": street["boxes"][:, 2].numpy().round(1), "y_max": street["boxes"][:, 3].numpy().round(1),
}).head(10)

In [ ]:
def show_detections(ax, image, prediction=None, target=None, score_threshold=0.5,
                    categories=None, title=None):
    # target boxes: green dashed; predictions above the threshold: red with "category score"
    ax.imshow(to_display(image))
    ax.axis("off")
    if target is not None:
        draw_boxes(ax, target["boxes"], color="lime", linestyle="--")
    if prediction is not None:
        keep = prediction["scores"] >= score_threshold
        scores = prediction["scores"][keep].tolist()
        if categories is None:
            names = [f"{s:.2f}" for s in scores]
        else:
            names = [f"{categories[c]} {s:.2f}" for c, s in zip(prediction["labels"][keep].tolist(), scores)]
        draw_boxes(ax, prediction["boxes"][keep], color="red", labels=names)
    if title:
        ax.set_title(title)


fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, name in zip(axes, demo_images):
    show_detections(ax, demo_images[name], demo_outputs[name], score_threshold=0.5,
                    categories=COCO_CATEGORIES, title=f"{name} (score >= 0.5)")
plt.tight_layout()
plt.show()

### 🎚️ The score threshold

The model returns every detection whose score is above `roi_heads.score_thresh` (0.05 by
default) — low-confidence guesses included. What we **display** or **act on** is a separate
decision: a higher threshold gives fewer false alarms but more missed objects.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for ax, threshold in zip(axes, [0.05, 0.5, 0.9]):
    n = int((street["scores"] >= threshold).sum())
    show_detections(ax, demo_images["street"], street, score_threshold=threshold,
                    categories=COCO_CATEGORIES, title=f"score >= {threshold}: {n} boxes")
plt.tight_layout()
plt.show()

### 🧪 Experiment: switch NMS off

The final, class-aware NMS of Faster R-CNN is controlled by `roi_heads.nms_thresh`. A box is
removed if its IoU with a better box of the same class is **above** this threshold — so with
`1.0`, nothing is ever removed. Let's see the raw duplicates. 👀

(The model never returns more than `roi_heads.detections_per_img` boxes, so without NMS the
count hits that cap.)

In [ ]:
print("score threshold         :", coco_detector.roi_heads.score_thresh)
print("NMS IoU threshold       :", coco_detector.roi_heads.nms_thresh)
print("max detections per image:", coco_detector.roi_heads.detections_per_img)

original_nms = coco_detector.roi_heads.nms_thresh
try:
    coco_detector.roi_heads.nms_thresh = 1.0                     # NMS removes nothing
    street_no_nms = detect(coco_detector, [demo_images["street"]])[0]
finally:
    coco_detector.roi_heads.nms_thresh = original_nms            # always restore the setting

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, out, title in [(axes[0], street_no_nms, "NMS off"), (axes[1], street, f"NMS on ({original_nms})")]:
    n = int((out["scores"] >= 0.5).sum())
    show_detections(ax, demo_images["street"], out, score_threshold=0.5, title=f"{title}: {n} boxes >= 0.5")
plt.tight_layout()
plt.show()

### ⚓ The anchors of this model

The proposal network uses these anchor sizes (in pixels of the **resized** image) and aspect
ratios ($h / w$). Below we draw the anchors of the first four sizes, all centred at one position.

In [ ]:
anchor_generator = coco_detector.rpn.anchor_generator
print("anchor sizes per feature map :", anchor_generator.sizes)
print("aspect ratios per feature map:", anchor_generator.aspect_ratios)

sizes = anchor_generator.sizes[0][:4]
ratios = anchor_generator.aspect_ratios[0]
cx, cy = 256, 256

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(skdata.astronaut())
colours = plt.cm.viridis(np.linspace(0, 0.9, len(sizes)))
for size, colour in zip(sizes, colours):
    for ratio in ratios:
        h, w = size * np.sqrt(ratio), size / np.sqrt(ratio)     # area stays size^2, h/w = ratio
        draw_boxes(ax, [[cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]], color=colour, linewidth=1.5)
ax.set_title(f"{len(sizes)} sizes x {len(ratios)} aspect ratios at ONE position")
ax.axis("off")
plt.show()

> 🔎 Anchors like these are placed at **every position of every feature map** — tens of
> thousands of candidates for one image. Most of them are background, which is why the
> proposal network keeps only the best ones (after its own NMS) for the second stage.

---
# 🚶 Part 5 — Fine-tuning on our own data: Penn-Fudan pedestrians

The [Penn-Fudan database](https://www.cis.upenn.edu/~jshi/ped_html/) contains **170 street
images** (from the University of Pennsylvania and Fudan University campuses) with **a pixel mask
for every annotated pedestrian**. We convert the masks to boxes.

COCO already has a `person` class with hundreds of thousands of annotated people, so the
pre-trained model is a **strong zero-shot baseline** — it finds pedestrians before we train
anything. Our question: **can 110 images and a few minutes of fine-tuning match it?**

The recipe mirrors ML #04: keep the pre-trained backbone, **swap the head** for our classes,
train. It is exactly the recipe you would use for classes that COCO has never seen — cracks,
blood cells, the objects of your own project — where no zero-shot baseline exists at all.

## 📂 The dataset class

Each mask image stores `0` for background and `1, 2, 3, …` for the individual pedestrians.
For each pedestrian id we build a binary mask, and `masks_to_boxes` gives us the box around it.

> ⚠️ `masks_to_boxes` returns the indices of the **last** pixels. A pixel with index $i$ covers
> the interval $[i, i+1)$, so we add 1 to `x_max` and `y_max` to get the box **edges**.

In [ ]:
class PennFudanDataset(Dataset):
    def __init__(self, image_files, transforms=None):
        self.image_files = list(image_files)
        self.transforms = transforms

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image_path = self.image_files[index]
        mask_path = image_path.parent.parent / "PedMasks" / f"{image_path.stem}_mask.png"

        image = read_image(str(image_path), mode=ImageReadMode.RGB)    # uint8 (3, H, W)
        mask = read_image(str(mask_path))[0]                          # (H, W): 0 = background, 1..K = people
        object_ids = torch.unique(mask)[1:]                           # drop the background id 0
        masks = mask[None] == object_ids[:, None, None]               # (K, H, W), one mask per person

        boxes = masks_to_boxes(masks)                                 # (K, 4) last-pixel indices
        boxes[:, 2:] += 1                                             # -> box edges

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=image.shape[-2:]),
            "labels": torch.ones(len(boxes), dtype=torch.int64),      # 1 = pedestrian, 0 = background
        }
        image = tv_tensors.Image(image)
        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target

### Train / validation / test 🔀

170 images is very little, so we split **110 / 30 / 30** with a fixed seed. Validation chooses
the training epoch and any settings; the test split is used **once**, at the very end.

In [ ]:
image_files = sorted((PENNFUDAN_DIR / "PNGImages").glob("*.png"))
perm = np.random.default_rng(SEED).permutation(len(image_files))

train_files = [image_files[i] for i in perm[:110]]
val_files   = [image_files[i] for i in perm[110:140]]
test_files  = [image_files[i] for i in perm[140:]]

print("train / val / test images:", len(train_files), len(val_files), len(test_files))

## 🔄 Augmentation that moves the boxes too

If we flip an image horizontally, **the boxes must be flipped as well** — otherwise the labels
are wrong. The `torchvision.transforms.v2` API does that automatically for `tv_tensors.BoundingBoxes`.

- training: random horizontal flip + conversion to float in $[0, 1]$,
- validation/test: only the conversion.

In [ ]:
train_transforms = T.Compose([T.RandomHorizontalFlip(p=0.5), T.ToDtype(torch.float32, scale=True)])
eval_transforms = T.ToDtype(torch.float32, scale=True)

ped_train = PennFudanDataset(train_files, train_transforms)
ped_val   = PennFudanDataset(val_files, eval_transforms)
ped_test  = PennFudanDataset(test_files, eval_transforms)

# The same image with and without a flip: the boxes follow the people
image, target = PennFudanDataset(train_files)[0]
flipped_image, flipped_target = T.RandomHorizontalFlip(p=1.0)(image, target)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
show_detections(axes[0], image, target=target, title=f"original: {len(target['boxes'])} pedestrians")
show_detections(axes[1], flipped_image, target=flipped_target, title="flipped: boxes flipped too")
plt.tight_layout()
plt.show()

print({k: (type(v).__name__, tuple(v.shape)) for k, v in target.items()})

In [ ]:
n_people = [len(PennFudanDataset([f])[0][1]["boxes"]) for f in image_files]
print("pedestrians in total     :", sum(n_people))
print("pedestrians per image    :", pd.Series(n_people).value_counts().sort_index().to_dict())

## 🧪 Zero-shot baseline: the COCO detector, `person` class only

We run the unchanged COCO detector on the validation images and keep only its `person`
detections. `predict_dataset` collects predictions and targets in the format our
`detection_report` from Part 1 expects.

In [ ]:
PERSON_ID = COCO_CATEGORIES.index("person")


def predict_dataset(model, dataset, keep_label=None):
    predictions, targets = [], []
    for image, target in dataset:
        pred = detect(model, [image])[0]
        if keep_label is not None:                         # e.g. only COCO "person"
            keep = pred["labels"] == keep_label
            pred = {k: v[keep] for k, v in pred.items()}
        predictions.append(pred)
        targets.append({"boxes": target["boxes"].as_subclass(torch.Tensor)})
    return predictions, targets


val_pred_zero_shot, val_targets = predict_dataset(coco_detector, ped_val, keep_label=PERSON_ID)
results = {"COCO zero-shot (person)": detection_report(val_pred_zero_shot, val_targets)}
pd.DataFrame(results).T.round(3)

## 🔁 Swap the head

Exactly as in ML #04 we keep the pre-trained body and replace the classifier — here the
`box_predictor` of the second stage:

- COCO head: 91 class scores + $91 \times 4$ box offsets (one box refinement **per class**),
- our head: **2** classes (`background`, `pedestrian`) + $2 \times 4$ box offsets.

`trainable_backbone_layers` decides how many backbone blocks are fine-tuned; the rest stays
frozen (default for this model: 3 of 6). The RPN and the rest of the second stage keep their
pre-trained weights and are trained too.

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  total parameters:     {total:,}")
    print(f"  trainable parameters: {trainable:,}")
    return total, trainable


def build_pedestrian_detector(trainable_backbone_layers=3, seed=SEED):
    torch.manual_seed(seed)                                 # reproducible initialisation of the new head
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=detector_weights,
                                              trainable_backbone_layers=trainable_backbone_layers)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
    return model.to(device)


print("COCO head:", coco_detector.roi_heads.box_predictor)
ped_detector = build_pedestrian_detector()
print("our head :", ped_detector.roi_heads.box_predictor)
count_params(ped_detector);

## 🔥 Training mode returns losses

torchvision detection models behave differently in the two modes:

- `model.eval()` + `model(images)` → **detections**,
- `model.train()` + `model(images, targets)` → a **dict of losses**.

The four losses are **two stages × (classification + box regression)** — the localizer idea from
Part 2, used twice:

| loss | stage | what it learns |
|---|---|---|
| `loss_objectness` | RPN | is there *any* object at this anchor? (binary cross-entropy) |
| `loss_rpn_box_reg` | RPN | anchor → proposal offsets (smooth L1) |
| `loss_classifier` | RoI head | which class is this proposal? (cross-entropy) |
| `loss_box_reg` | RoI head | proposal → final box offsets (smooth L1) |

Images in a batch have **different sizes**, so we cannot stack them into one tensor. The
`collate_detection` function keeps a batch as a **list** of images and a list of targets.

In [ ]:
def collate_detection(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


ped_train_loader = DataLoader(ped_train, batch_size=4, shuffle=True, collate_fn=collate_detection,
                              generator=torch.Generator().manual_seed(SEED))

images, targets = next(iter(ped_train_loader))
print("image shapes:", [tuple(img.shape) for img in images])

ped_detector.train()
loss_dict = ped_detector([img.to(device) for img in images],
                         [{k: v.to(device) for k, v in t.items()} for t in targets])
{k: round(v.item(), 4) for k, v in loss_dict.items()}

## ⭐ The training loop

- **Optimizer:** SGD with momentum, the usual choice in torchvision's detection references.
- **Loss:** the sum of the four losses.
- **Model selection:** after every epoch we compute AP50 on the **validation** set and restore
  the best epoch at the end.

> ⏱️ On a recent laptop CPU one epoch took us about 1.5 minutes (5 epochs ≈ 8 minutes); a GPU
> runtime is much faster. GPU training is not bit-for-bit reproducible, so your numbers can
> differ slightly.

In [ ]:
def train_detector(model, train_loader, val_dataset, epochs=5, lr=0.01, seed=SEED):
    torch.manual_seed(seed)
    if train_loader.generator is not None:
        train_loader.generator.manual_seed(seed)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))

    step_losses, epoch_rows = [], []
    best_ap, best_state, best_epoch = -1.0, None, None
    for epoch in range(epochs):
        model.train()
        for images, targets in tqdm(train_loader, desc=f"epoch {epoch + 1}/{epochs}", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()
            step_losses.append({k: v.item() for k, v in loss_dict.items()})

        predictions, targets_val = predict_dataset(model, val_dataset)
        report = detection_report(predictions, targets_val)
        epoch_rows.append({"epoch": epoch + 1, **report})
        print(f"epoch {epoch + 1}: " + ", ".join(f"{k} = {v:.3f}" for k, v in report.items()))
        if report["AP50"] > best_ap:
            best_ap, best_epoch = report["AP50"], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    print(f"restored epoch {best_epoch} (validation AP50 = {best_ap:.3f})")
    return pd.DataFrame(step_losses), pd.DataFrame(epoch_rows)

In [ ]:
%%time
ped_detector = build_pedestrian_detector()
ped_losses, ped_epochs = train_detector(ped_detector, ped_train_loader, ped_val, epochs=5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for col in ped_losses.columns:
    ax[0].plot(ped_losses[col].rolling(10, min_periods=1).mean(), label=col)
ax[0].set_xlabel("training step"); ax[0].set_title("training losses (rolling mean of 10 steps)")
ax[0].legend()

for col in ["AP50", "AP75", "AP@[.50:.95]"]:
    ax[1].plot(ped_epochs["epoch"], ped_epochs[col], "o-", label=col)
ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1.02); ax[1].set_title("validation"); ax[1].legend()
plt.tight_layout()
plt.show()

## 📊 Fine-tuned vs. zero-shot

Both models are evaluated on the same 30 validation images with the same AP code. The
validation set also chose the epoch, so these are **development** results — the untouched test
set comes at the very end.

In [ ]:
val_pred_finetuned, _ = predict_dataset(ped_detector, ped_val)
results["fine-tuned (pedestrian)"] = detection_report(val_pred_finetuned, val_targets)
pd.DataFrame(results).T.round(3)

### 🤔 Did fine-tuning win?

Expect the two rows to be **close** — and do not be surprised if the zero-shot model is ahead.

1. **The evaluation set is tiny.** 30 images with about 90 pedestrians: one or two boxes more
   or less can move AP by a point or more. Differences of that size are **within noise** —
   another split could reverse them.
2. **The starting points are very different.** The zero-shot head learned `person` from
   hundreds of thousands of examples; our new head started from **random weights** and saw 110
   images for a few minutes. Getting this close is the real transfer-learning result.
3. **Baseline first.** If a pre-trained model already covers your class, evaluate it before
   you train anything — it may be all you need. Fine-tuning pays off for **new classes**, a
   **different domain** (thermal cameras, top-down drone views, microscopy) or a **different
   annotation policy**.

Activity 4 asks you to beat the baseline anyway. 🏆

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, i in zip(axes.flat, range(6)):
    image, _ = ped_val[i]
    show_detections(ax, image, val_pred_finetuned[i], val_targets[i], score_threshold=0.5,
                    title=f"{val_files[i].name}")
plt.suptitle("fine-tuned model on validation images — green dashed = truth, red = prediction (score >= 0.5)")
plt.tight_layout()
plt.show()

## 🕵️ Error analysis

AP summarises the whole ranked list. To **act** on a detector we must pick an operating point —
here a score threshold of 0.5 — and look at the concrete mistakes:

- **FP** — a predicted box without a matching pedestrian (IoU ≥ 0.5),
- **FN** — a pedestrian without a matching prediction.

In [ ]:
def count_errors(prediction, target, score_threshold=0.5, iou_threshold=0.5):
    keep = prediction["scores"] >= score_threshold
    is_tp = match_predictions(prediction["boxes"][keep], prediction["scores"][keep],
                              target["boxes"], iou_threshold)
    tp = int(is_tp.sum())
    return {"TP": tp, "FP": int(keep.sum()) - tp, "FN": len(target["boxes"]) - tp}


errors = pd.DataFrame([{"image": f.name, **count_errors(p, t)}
                       for f, p, t in zip(val_files, val_pred_finetuned, val_targets)])
totals = errors[["TP", "FP", "FN"]].sum()
print(totals.to_dict())
print(f"precision = {totals.TP / (totals.TP + totals.FP):.3f}, recall = {totals.TP / (totals.TP + totals.FN):.3f}")

errors.assign(errors=errors.FP + errors.FN).sort_values("errors", ascending=False).head(5)

In [ ]:
worst = errors.assign(errors=errors.FP + errors.FN).sort_values("errors", ascending=False).index[:3]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, i in zip(axes, worst):
    image, _ = ped_val[i]
    row = errors.loc[i]
    show_detections(ax, image, val_pred_finetuned[i], val_targets[i], score_threshold=0.5,
                    title=f"{row.image}: TP {row.TP}, FP {row.FP}, FN {row.FN}")
plt.tight_layout()
plt.show()

> 🔎 **Look closely before you blame the model.** Typical failure cases are **small, distant
> people** (remember the IoU-vs-size plot from Part 1), people **occluded** by other people or
> **cut off** by the image border, and groups. Some "false positives" may be people that are
> only partly visible and annotated differently than you would expect — the **annotation
> policy** is part of the task definition. Write down two concrete failure patterns you see.

## 📌 Summary

| Concept | One-line takeaway |
|---|---|
| Box formats | `xyxy`, `xywh`, `cxcywh` — always check which one a tool expects |
| IoU | overlap / union; the same pixel error hurts small boxes more |
| NMS | greedy: keep the best box, remove same-class boxes with IoU above the threshold |
| AP | area under the precision–recall envelope; missed objects cap it |
| Localization | one CNN, two heads: $\mathcal{L}_{\text{cls}} + \lambda\,\mathcal{L}_{\text{box}}$ — but only one object |
| Detection | many candidates (anchors / grid / queries) + a way to remove duplicates |
| Fine-tuning | swap the box predictor, train with the four losses, select on validation AP |

---
# Activities ✏️

Activity 1 is in Part 1 (pen and paper). The remaining activities include starter code and
empty cells for your solutions. Use the **validation** set for every comparison and decision;
the test set is used only in the final cell of the notebook.

## ✏️ Activity 2 — Balance the two losses ⚖️

The localizer was trained with $\lambda = 5$ in $\mathcal{L}_{\text{cls}} + \lambda\,\mathcal{L}_{\text{box}}$.

**Your task:**

1. **Predict first:** what happens to accuracy and mean IoU for $\lambda = 0$? And for a very large $\lambda$?
2. Train the localizer with $\lambda \in \{0, 1, 5, 25, 100\}$ (same seed and epochs) and collect
   the final **validation** metrics in one table.
3. Plot accuracy and mean IoU against $\lambda$ (log-scaled x axis; plot $\lambda = 0$ separately
   or as a horizontal line). Which value would you choose, and why?
4. Did a large $\lambda$ hurt the classification? If not, why can the trade-off stay hidden on
   this dataset — and how could you change the data to make it visible?

In [ ]:
# Starter: train_localizer(box_weight=...) returns (model, history); history.iloc[-1] holds the final validation metrics

## ✏️ Activity 3 — Choosing the thresholds 🎚️

Two post-processing settings decide what the detector finally outputs:
`roi_heads.score_thresh` and `roi_heads.nms_thresh`.

**Your task** (fine-tuned model, **validation** images):

1. For `nms_thresh` ∈ {0.3, 0.5, 0.7, 0.9} compute AP50 (keep `score_thresh = 0.05`).
2. For score thresholds ∈ {0.1, 0.3, 0.5, 0.7, 0.9} compute **precision and recall** with
   `count_errors` (keep `nms_thresh = 0.5`). Plot recall against precision.
3. Also compute AP50 with `score_thresh` = 0.5 and 0.9. Compare: how strongly does the score
   threshold change **AP**, and how strongly does it change **precision and recall**? Explain
   the difference using the ranked list from Part 1. Which threshold would you choose for
   (a) an emergency-braking system, (b) counting people in a shop?
4. Reset both settings to their original values at the end.

In [ ]:
def set_postprocessing(model, score_thresh=0.05, nms_thresh=0.5):
    model.roi_heads.score_thresh = score_thresh
    model.roi_heads.nms_thresh = nms_thresh

# Example: set_postprocessing(ped_detector, nms_thresh=0.3); preds, tgts = predict_dataset(ped_detector, ped_val)

## ✏️ Activity 4 — Can you beat the zero-shot baseline? 🏆

Use fresh models from `build_pedestrian_detector` and collect **validation** AP50 and
AP@[.50:.95] in one table together with the zero-shot row.

1. **Keep what COCO knows about people ("head surgery").** Our new head started from random
   weights. The starter cell instead copies the COCO head's rows for `background` and `person`
   into the new two-class head. Evaluate this model **before any training** and compare it with
   the zero-shot row. Why are the numbers (almost) the same?
2. Fine-tune the initialised model with the default `lr=0.01` and with a **10× smaller**
   learning rate. Look at the AP after the **first** epoch. Which lesson from ML #04 about
   fine-tuning do you see again?
3. **How much data?** Repeat your best setting with only the first **10** and **30** images of
   `train_files` (a new `PennFudanDataset` and `DataLoader` for each size).
4. **Is a difference real?** About 90 pedestrians are in the validation set. How much does recall
   change when one more pedestrian is missed? How large must an AP difference be before you
   trust it?

*Extension:* compare `trainable_backbone_layers` = 0, 3 and 6.

> ⏱️ Every run is a full training — use a GPU runtime, or fewer epochs on a CPU.

In [ ]:
def init_head_from_coco(model, coco_model, class_ids=(0, PERSON_ID)):
    # Copy the COCO head's rows for the chosen classes (background first) into the new head
    old, new = coco_model.roi_heads.box_predictor, model.roi_heads.box_predictor
    keep = torch.tensor(class_ids)
    d = old.bbox_pred.in_features
    with torch.no_grad():
        new.cls_score.weight.copy_(old.cls_score.weight[keep])
        new.cls_score.bias.copy_(old.cls_score.bias[keep])
        # bbox_pred has 4 outputs per class: (classes * 4, d) -> (classes, 4, d) -> pick -> flatten
        new.bbox_pred.weight.copy_(old.bbox_pred.weight.view(-1, 4, d)[keep].reshape(-1, d))
        new.bbox_pred.bias.copy_(old.bbox_pred.bias.view(-1, 4)[keep].reshape(-1))
    return model

# Example:
# surgery_model = init_head_from_coco(build_pedestrian_detector(), coco_detector)
# detection_report(*predict_dataset(surgery_model, ped_val))

## ✏️ Activity 5 — A transformer detector: DETR 🤖

DETR (*DEtection TRansformer*) predicts a **fixed set** of (class, box) pairs — one per learned
**object query** — and is trained with a one-to-one (Hungarian) matching between predictions and
ground-truth objects. Each object is matched to exactly one query, so the model learns not to
output duplicates and needs **no anchors and no NMS**.

**Your task:**

1. Run the starter cell (it downloads about 170 MB) and draw DETR's detections on the four demo
   images from Part 4. Compare them with Faster R-CNN.
2. How many object queries does the model have (`detr.model.config.num_queries`)? What happens
   to the "unused" queries?
3. Evaluate DETR's zero-shot `person` detections on the **validation** images with our
   `detection_report` (use a low threshold, e.g. 0.05, to get a full ranked list) and add the row
   to the results table.

> 💡 The pipeline returns boxes as dicts `{"xmin", "ymin", "xmax", "ymax"}` in pixels and labels
> as names. It expects PIL images: convert with `torchvision.transforms.functional.to_pil_image`.
> DETR's ResNet backbone comes from the `timm` library, which the cell installs. A load report
> listing unused `num_batches_tracked` keys is expected (BatchNorm counters the model does not use).
> If DETR does not load in your environment, `PekingU/rtdetr_r18vd` (RT-DETR, also NMS-free)
> works through the same pipeline without `timm`.

In [ ]:
%pip install -q "transformers>=4.41" timm

from transformers import pipeline
from torchvision.transforms.functional import to_pil_image

detr = pipeline("object-detection", model="facebook/detr-resnet-50",
                device=0 if torch.cuda.is_available() else -1)

detr_out = detr(to_pil_image(demo_images["street"]), threshold=0.5)
detr_out[:3]

---
## 🏁 Final held-out evaluation — run after all choices are frozen

Run this cell **once**, after you have chosen the model and its settings on the validation set.
It reports the zero-shot baseline and the fine-tuned model on the 30 test images. With so few
images, a difference of a few AP points is within noise — treat it as a rough estimate.

In [ ]:
set_postprocessing(ped_detector, score_thresh=0.05, nms_thresh=0.5)   # the settings used for validation

test_pred_zero_shot, test_targets = predict_dataset(coco_detector, ped_test, keep_label=PERSON_ID)
test_pred_finetuned, _ = predict_dataset(ped_detector, ped_test)

pd.DataFrame({
    "COCO zero-shot (person)": detection_report(test_pred_zero_shot, test_targets),
    "fine-tuned (pedestrian)": detection_report(test_pred_finetuned, test_targets),
}).T.round(3)